# Impact de la définition des fichiers réseaux

Ce notebook cherche à visualiser l'impact de la définition de l'offre sur le comportement de simulations MnMS.

## 1 - Environnement

### 1.1 - Dépendances

In [ ]:
# pip

%pip install --upgrade pip

In [ ]:
# installation des dépendances

%pip install --upgrade os
%pip install --upgrade sys

%pip install --upgrade pathlib

%pip install --upgrade importlib

%pip install --upgrade time

%pip install --upgrade matplotlib.pyplot

%pip install --upgrade numpy
%pip install --upgrade pandas
%pip install --upgrade geopandas

%pip install --upgrade contextily

%pip install --upgrade logging

In [ ]:
# chargement des dépendances

import os
import sys

from pathlib import Path

import importlib

import time

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import geopandas as gpd

import contextily as cx

from shapely.geometry import Point
from shapely import wkt
from shapely.ops import unary_union
from shapely.geometry import shape


import logging

In [ ]:
# configuration du logging

logging.basicConfig(
    level=logging.INFO,  
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

In [ ]:
# chemin relatif depuis ce notebook jusqu'au dossier parent de modules 

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

### 1.2 - Classes

In [ ]:
# import des classes

import modules.Utils as Utils
import modules.CopertEstimator
import modules.DemandVariationCarSimulation1Res

importlib.reload(Utils)
importlib.reload(sys.modules['modules.CopertEstimator'])
importlib.reload(sys.modules['modules.DemandVariationCarSimulation1Res'])

CopertEstimator = modules.CopertEstimator.CopertEstimator
DemandVariationCarSimulation1Res = modules.DemandVariationCarSimulation1Res.DemandVariationCarSimulation1Res

## 2 - Contexte, chargement, prétraitement

### 2.1 - Contexte

Une simulation MnMS est effectuée à partir de la description d'un réseau. Cette description prend la forme de trois fichiers : le graphe du réseau, la couche origine-destination et la couche de transfert. Le graphe du réseau décrit ses routes et carrefours comme un graphe orienté et pondéré. Ici, le graphe d'Amsterdam ne sera pas à construire mais directement repris de simulations précédentes effectuées sur Matsim. En revanche, la couche origine-destination et de transfert, grilles dont le rôle est de servir d'interface entre données réelles et graphe, ne sont pas fournis. Ces deux fichiers peuvent être construits à partir de méthodes MnMS en fonction de divers paramètres : le nombre de noeuds pas lignes et par colonnes et la distance maximale du rayon de recherche de noeuds sur le graphe à connecter avec les couches "interfaces". Dans les parties suivantes sont testés un certain nombre de ces fichiers "interfaces", chacun avec sa combinason de valeurs spécifiques pour chaque argument, dans le cadre d'une simulation MnMS sur le fichier de demande originale AmsterdamCity_7-9 afin de rendre compte de l'impact des paramètres sur les sorties d'une simulation. 

### 2.2 - Chargement et prétraitement

In [ ]:
# fichiers de demande

chemin_demande_matsim = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00.csv"
chemin_demande_amsterdam = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00.csv"
chemin_dossier_demande_amsterdam = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original"

In [ ]:
# fichier graphe

chemin_graphe = "../../network/graph/Amsterdam__Matsim__graph.json"

## 3 - Création de fichiers réseaux

In [ ]:
# génération d'od layer et de transit link pour divers nombre de noeuds

nx_interval = np.arange(14,30,2)
ny_interval = np.arange(14,30,2)
dist_connection_interval = np.arange(900,1200,100)

DVCS1R = DemandVariationCarSimulation1Res(chemin_dossier_demande_amsterdam, chemin_graphe)

for nx in nx_interval : 
    for ny in ny_interval : 
        for dist_connection in dist_connection_interval : 
            DVCS1R.create_network_files(nx, ny, dist_connection, "../../network/test_files")

## 4 - Simulations

In [ ]:
# simulations

nx_interval = np.arange(14,30,2)
ny_interval = np.arange(14,30,2)
dist_connection_interval = np.arange(900,1200,100) 
graph_name = Path(chemin_graphe).stem
a = 2.8981940480441857
b = -0.00010553060526140915
MFD_variables = [a, b]

DVCS1R = DemandVariationCarSimulation1Res(chemin_dossier_demande_amsterdam, chemin_graphe)

for nx in nx_interval : 
    for ny in ny_interval : 
        for dist_connection in np.arange(1000,1200,100) : 
            od_layer_path = f"../../network/test_files/{graph_name}_od_layer_{nx}_{ny}_{dist_connection}.json"
            transit_link_path = f"../../network/test_files/{graph_name}_transit_link_{nx}_{ny}_{dist_connection}_grid.json"
            output_directory = f"../../outputs/network_files/simulations/AmsterdamCity__Original_Original__Simulations_Dummy__{nx}_{ny}_{dist_connection}"
            DVCS1R.simulate(od_layer_path, transit_link_path, MFD_variables, "Dummy", output_directory)

## 5 - Analyse comparative

In [ ]:
# comparaisond des MDFs

Utils.MFD_comparison("../../outputs/network_files/simulations")

In [ ]:
# fonctions de creation d'un tableau pour l'analyse reseaux/copert

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

def tab_network_copert(directory):
    """
    Creates and return an dataframe with network specificities and copert estimation values from a given directory.

    Parameters
    ----------
    directory : str
        Directory from which simulation outputs file are taken.

    Returns
    -------
    pandas.DataFrame
    """

    # check
    if not directory or directory.strip() == "" : 
        raise ValueError("Null or invalid directory.")

    # List top-level scenario directories
    scenario_dirs = [
        f.path
        for f in os.scandir(directory)
        if f.is_dir() and not f.name.startswith('.')
    ]


    sensitivity_df = []

    for i, scenario_path in enumerate(scenario_dirs):

        # parameters extraction
        
        folder_name = os.path.basename(scenario_path)
        print(f"{scenario_path}")
        last_part = folder_name.split("__")[-1]
        parameters = last_part.split("_")
        parameters = [int(x) for x in parameters]


        subfolder = [f.path for f in os.scandir(scenario_path) if f.is_dir()]

        CE = CopertEstimator(subfolder[0], "epsg:32631", "../../data/copert_data/test/FE_COPERT2025.csv")
        CE.copert_estimation()

        row = {"NX":parameters[0], "NY":parameters[1], "DIST_CONNECTION":parameters[2], "NOX":CE.get_nox(), "CO2":CE.get_co2()}
        
        if len(sensitivity_df) == 0 :
            sensitivity_df = pd.DataFrame([row])
        else : 
            sensitivity_df = pd.concat([sensitivity_df, pd.DataFrame([row])], ignore_index=True)

    return sensitivity_df.copy()

def network_copert_analysis_plot_horizontal(df):
    """
    Affichage horizontal :
    - Gauche : NOX en fonction de NX, NY, DIST_CONNECTION
    - Droite : CO2 en fonction de NX, NY, DIST_CONNECTION
    """

    fig = plt.figure(figsize=(16, 6))

    # --- Plot NOX ---
    ax1 = fig.add_subplot(121, projection='3d')
    sc1 = ax1.scatter(df["NX"], df["NY"], df["DIST_CONNECTION"],
                      c=df["NOX"], cmap="coolwarm")

    ax1.set_title("NOX en fonction de NX, NY, DIST_CONNECTION")
    ax1.set_xlabel("NX")
    ax1.set_ylabel("NY")
    ax1.set_zlabel("DIST_CONNECTION")

    # --- Plot CO2 ---
    ax2 = fig.add_subplot(122, projection='3d')
    sc2 = ax2.scatter(df["NX"], df["NY"], df["DIST_CONNECTION"],
                      c=df["CO2"], cmap="coolwarm")

    ax2.set_title("CO2 en fonction de NX, NY, DIST_CONNECTION")
    ax2.set_xlabel("NX")
    ax2.set_ylabel("NY")
    ax2.set_zlabel("DIST_CONNECTION")

    # --- Légende commune ---
    cbar = fig.colorbar(sc1, ax=[ax1, ax2], location='right', shrink=0.75)
    cbar.set_label("Valeur (NOX / CO2)")

    plt.tight_layout()
    plt.show()

  

In [ ]:
# analyse 

sensitivity_df = tab_network_copert("../../outputs/network_files/simulations")

In [ ]:
# sauvegarde

sensitivity_df.to_csv("../../outputs/network_files/analysis/sensitivity/sensitivity.csv", sep=';', index=False)

In [ ]:
# affichage

network_copert_analysis_plot_horizontal(sensitivity_df)

## 6 - Bilan

La définition des couches origines-destination et de transit a un impact important sur la simulation et donc sur l'estimation des émissions. Cette influence doit être prise en compte dans le cadre de travaux dont l'objectif est de réaliser des systèmes d'estimation ou de prédiction des émissions. Cependant une analyse de sensibilité est l'étude du relatif : seul le fait d'avoir un matériaux commun et unique est essentiel, le degré de véracité de valeurs n'a pas d'influence sur l'analyse de leurs fluctuations en fonction de l'évolution de paramètres. Il pourrait être intéressant de réaliser une analyse plus profonde l'impact de la définition du réseau (donc de l'offre) sur le comportement de simualtions MnMS et donc sur l'estimationd des émissions.